In [ ]:
!pip install mintpy hyp3-sdk rasterio geopandas

In [ ]:
import hyp3_sdk as sdk
import zipfile
import os

# Your direct credentials
username = "taruprajapati8@gmail.com"
password = "Whisperverse@11"

hyp3 = sdk.HyP3(username=username, password=password)

jobs = hyp3.find_jobs(name='cern_insar')
print(f"Found {len(jobs)} jobs. Downloading...")

os.makedirs('data', exist_ok=True)
downloaded_files = jobs.download_files('data')

os.makedirs('data/unzipped', exist_ok=True)
for file in os.listdir('data'):
    if file.endswith('.zip'):
        with zipfile.ZipFile(os.path.join('data', file), 'r') as zip_ref:
            zip_ref.extractall('data/unzipped')

print("All pairs unzipped successfully!")


In [ ]:
import os

# 1. Clean out the corrupted intermediate HDF5 files from the interrupted run
!rm -rf inputs
!rm -f smallbaselineApp.cfg mintpy_config.txt *.h5

# 2. Re-index HyP3 stack
!prep_hyp3.py -i "./data/unzipped/*"

# 3. Create fresh robust configuration
config_content = """# MintPy HyP3 Configuration
mintpy.load.processor      = hyp3
mintpy.load.unwFile        = ./data/unzipped/*/*unw_phase.tif
mintpy.load.corFile        = ./data/unzipped/*/*corr.tif
mintpy.load.demFile        = ./data/unzipped/*/*dem.tif
mintpy.load.incAngleFile   = ./data/unzipped/*/*inc_map.tif
mintpy.load.waterMaskFile  = none

# Disable network dependencies that fail on connection drop
mintpy.troposphericDelay.method = no
mintpy.topographicResidual    = no
mintpy.network.coherenceBased = no
mintpy.plot                   = no

# Resource options
mintpy.compute.cluster     = local
mintpy.compute.numWorker   = 4
"""

with open('mintpy_config.txt', 'w') as f:
    f.write(config_content)

print("Cleared broken files and created clean config!")

# 4. Run smallbaselineApp from scratch
!smallbaselineApp.py mintpy_config.txt

In [ ]:
# 1. Display the velocity map image directly in Colab
from IPython.display import Image

# Generate the velocity PNG plot
!view.py velocity.h5 --notitle --vlim -2 2 --colormap jet -o velocity_map.png

# Display image
Image('velocity_map.png')

In [ ]:
from IPython.display import Image

# Plot the primary velocity map output
!view.py velocity.h5 --vlim -2 2 --colormap jet -o velocity_map.png

# Display the exported PNG
Image('velocity_map.png')

In [ ]:
from IPython.display import Image

# 1. Plot the final cumulative displacement (units in cm)
# Selecting date=-1 automatically picks the latest date in your time-series
!view.py timeseries.h5 date=-1 --vlim -5 5 --colormap jet --unit cm -o final_cumulative_displacement.png

# 2. Display the generated image in Colab
Image('final_cumulative_displacement.png')

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# 1. Open the time-series HDF5 file
with h5py.File('timeseries.h5', 'r') as f:
    # Extract displacement array (Convert from meters to centimeters)
    timeseries_data = f['timeseries'][:] * 100

    # Get acquisition dates
    dates = [d.decode('utf-8') for d in f['date'][:]]

# Cumulative displacement is the LAST date slice (index -1)
final_cum_disp = timeseries_data[-1, :, :]

start_date = dates[0]
end_date = dates[-1]

# 2. Plot the cumulative displacement map
plt.figure(figsize=(12, 8))
plt.imshow(final_cum_disp, cmap='jet', vmin=-5, vmax=5)

cbar = plt.colorbar(shrink=0.8)
cbar.set_label('Cumulative Displacement (cm)', rotation=270, labelpad=15)

plt.title(f'Final Cumulative Displacement Map\n({start_date} to {end_date})', fontsize=14)
plt.axis('off')

# Save and show image
output_filename = 'final_cumulative_displacement.png'
plt.savefig(output_filename, dpi=300, bbox_inches='tight')
plt.show()

print(f"Successfully created and saved {output_filename}!")

In [ ]:
# Convert velocity.h5 to standard GeoTIFF format
!save_gdal.py velocity.h5 -o velocity.tif

In [ ]:
import h5py
import numpy as np
from osgeo import gdal, osr

# 1. Read metadata and final date slice from timeseries.h5
with h5py.File('timeseries.h5', 'r') as f:
    # Get last date slice (convert to float32)
    data = f['timeseries'][-1, :, :].astype(np.float32)

    # Extract georeferencing metadata
    meta = dict(f.attrs)

# Get dimensions
rows, cols = data.shape

# 2. Extract spatial reference information
# MintPy stores bounding box / transform in metadata
x0 = float(meta.get('X_FIRST', 220200.0))
y0 = float(meta.get('Y_FIRST', 5247800.0))
dx = float(meta.get('X_STEP', 80.0))
dy = float(meta.get('Y_STEP', -80.0))
epsg_code = int(meta.get('EPSG', 32632))

# 3. Write out GeoTIFF using GDAL
driver = gdal.GetDriverByName('GTiff')
output_file = 'final_cumulative_displacement.tif'
dataset = driver.Create(output_file, cols, rows, 1, gdal.GDT_Float32)

# Set geotransform and projection
dataset.SetGeoTransform((x0, dx, 0, y0, 0, dy))

srs = osr.SpatialReference()
srs.ImportFromEPSG(epsg_code)
dataset.SetProjection(srs.ExportToWkt())

# Write raster data
dataset.GetRasterBand(1).WriteArray(data)
dataset.FlushCache()
dataset = None

print(f"GeoTIFF successfully created and saved as '{output_file}'!")

In [ ]:
# Zip all deliverable files together
!zip insar_results.zip final_cumulative_displacement.tif final_cumulative_displacement.png velocity.tif velocity.kmz

print("Created insar_results.zip! Right-click 'insar_results.zip' in the Files tab to download.")

In [ ]:
from google.colab import files

# Triggers direct browser download for the zip file
files.download('insar_results.zip')

In [ ]:
from google.colab import drive
import shutil

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Copy config and script files to Drive
shutil.copy('mintpy_config.txt', '/content/drive/MyDrive/mintpy_config.txt')

print("Configuration backed up to Google Drive!")

In [ ]:
import os

# Files and logs to collect
files_to_zip = [
    'mintpy_config.txt',
    'smallbaselineApp.cfg',
    'reference_date.txt',
    'pic',  # folder containing diagnostic figures/plots
    'inputs/geometryGeo.h5' # geometry metadata (if exists)
]

# Filter to files that exist on disk
existing_files = [f for f in files_to_zip if os.path.exists(f)]

# Zip them together
cmd = f"zip -r project_metadata_and_configs.zip {' '.join(existing_files)}"
os.system(cmd)

print("Created 'project_metadata_and_configs.zip'!")

In [ ]:
from google.colab import files

# Triggers download of the configuration & metadata zip
files.download('project_metadata_and_configs.zip')

In [ ]:
from google.colab import files

# Download the multi-panel plot image (velocity, velocityStd, intercept, etc.)
files.download('/content/velocity_map.png')

In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Output
import ipywidgets as widgets

# 1. Load the dataset from timeseries.h5
with h5py.File('timeseries.h5', 'r') as f:
    # Convert displacements from meters to cm
    ts_data = f['timeseries'][:] * 100
    dates = [d.decode('utf-8') for d in f['date'][:]]

dates_dt = pd.to_datetime(dates, format='%Y%m%d')
cum_disp = ts_data[-1, :, :]  # Final cumulative slice
rows, cols = cum_disp.shape

# 2. Interactive Widget setup
row_slider = IntSlider(min=0, max=rows-1, step=1, value=rows//2, description='Row (Y):')
col_slider = IntSlider(min=0, max=cols-1, step=1, value=cols//2, description='Col (X):')

out = Output()

def update_plots(row, col):
    with out:
        out.clear_output(wait=True)

        # Create side-by-side plots
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

        # Panel 1: Map with Selected Point Highlighted
        im = ax1.imshow(cum_disp, cmap='jet', vmin=-5, vmax=5)
        ax1.plot(col, row, 'k*', markersize=12, markeredgecolor='white', label='Selected Point')
        ax1.set_title(f'Cumulative Displacement Map\nSelected Pixel: ({col}, {row})')
        ax1.set_xlabel('Column')
        ax1.set_ylabel('Row')
        ax1.legend(loc='upper right')
        plt.colorbar(im, ax=ax1, label='Cumulative Disp (cm)', shrink=0.8)

        # Panel 2: Displacement Time-Series
        point_series = ts_data[:, row, col]
        ax2.plot(dates_dt, point_series, 'o-', color='#d62728', linewidth=2, markersize=5)
        ax2.axhline(0, color='black', linestyle='--', alpha=0.5)
        ax2.set_title(f'Displacement History for Pixel ({col}, {row})')
        ax2.set_xlabel('Date')
        ax2.set_ylabel('Displacement (cm)')
        ax2.grid(True, linestyle=':', alpha=0.6)
        fig.autofmt_xdate()

        plt.tight_layout()
        plt.show()

# Connect sliders to update function
widgets.interactive_output(update_plots, {'row': row_slider, 'col': col_slider})

display(widgets.VBox([row_slider, col_slider]), out)

In [ ]:
import os
from google.colab import files

# 1. Zip timeseries.h5 (zipping makes the download much faster)
!zip timeseries.zip timeseries.h5

# 2. Download it straight to your laptop
files.download('timeseries.zip')